In [2]:
import io
import base64
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from matplotlib.ticker import FormatStrFormatter, FixedLocator, MaxNLocator
from scipy.signal import find_peaks

# =============================
# 0. GLOBAL STYLE
# =============================

plt.close("all")
plt.rcParams["xtick.direction"] = "out"
plt.rcParams["ytick.direction"] = "out"

display(HTML("""
<style>
.jp-OutputArea-output, .widget-label, .widget-html-content {
    font-size: 14px;
}

.widget-text input {
    color: var(--jp-ui-font-color1) !important;
    background-color: var(--jp-layout-color1) !important;
}

.widget-text input:disabled {
    color: var(--jp-ui-font-color1) !important;
    opacity: 1 !important;
}

.download-row {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    align-items: center;
    margin-top: 6px;
}

.download-btn {
    display: inline-block;
    padding: 8px 14px;
    border-radius: 6px;
    text-decoration: none;
    font-weight: 500;
    color: white !important;
    border: none;
    cursor: pointer;
    user-select: none;
}

.download-btn.orange {
    background: #d97706;
}

.download-btn.gray {
    background: #374151;
}

.download-btn.blue {
    background: #2563eb;
}
</style>
"""))

# =============================
# 1. FILE / APP STATE
# =============================

current_working_folder = Path.cwd().resolve()

selected_file_path = None
selected_file_name = ""
selected_folder_name = ""
selected_display_name = ""

f = None
peak = None
mass = None
spectrum = None

scan_size = 25.4
n_frames_total = 1
n_elements = 1

default_analysis_frame_start = 1
default_analysis_frame_end = 1
analysis_frame_start = 1
analysis_frame_end = 1
n_frames = 1
frames = np.arange(1, n_frames + 1)

base_name = ""
polarity = "Unknown"
possible_ion_map = {}
peak_table = pd.DataFrame()
table_title = "Table of Possible Ions"

cm_to_inch = 1 / 2.54
save_height = 10 * cm_to_inch
save_width_depth = 15.6 * cm_to_inch
save_width_tp = 12.6 * cm_to_inch
save_width_fp = 12.6 * cm_to_inch

# cache
depth_profile_cache = {}
projection_cache = {}

# =============================
# 2. ION MAPS FOR TABLE
# =============================

possible_ion_map_positive = {
    1: "H+",
    10: "B+",
    11: "B+",
    12: "C+",
    13: "CH+",
    14: "CH2+ / N+",
    15: "CH3+",
    16: "O+",
    17: "OH+",
    18: "H2O+ / NH4+",
    19: "H3O+ / F+ candidate",
    23: "Na+",
    24: "Mg+",
    25: "MgH+",
    26: "C2H2+",
    27: "Al+ / C2H3+",
    28: "Si+ / CO+ / C2H4+",
    29: "SiH+ / CHO+ / C2H5+",
    30: "NO+ / CH2O+",
    31: "P+ / CH3O+",
    32: "O2+",
    34: "S+ / H2S+",
    39: "K+",
    40: "Ca+ / Ar+",
    47: "PO+ / POH+",
    48: "Ti+ / SO+ / PO2H+",
    49: "PO2+ / HSO+",
    50: "Cr+ / TiO+ / V+",
    51: "Cr isotope / CrH+",
    52: "Cr+",
    53: "CrH+ / Cr isotope / FeO+",
    54: "Fe+ / CrO+ / Fe isotope",
    55: "Mn+ / FeH+ / CrOH+",
    56: "Fe+ / CaO+",
    57: "FeH+ / Fe isotope / Ni+ fragment / CaOH+",
    58: "Ni+ / FeO+",
    59: "Co+ / NiH+",
    60: "Ni+ / FeO+ / metal-oxide fragment",
    61: "NiH+ / CrO2+",
    62: "Ni isotope / FeO2+",
    63: "Cu+ isotope / PO2- candidate",
    64: "NiO+ / SO2+ / Zn isotope",
    65: "Cu+ / ZnH+ / HSO2+",
    66: "Zn+ / FeOH+",
    67: "ZnH+ / CrO2H+",
    68: "FeO+ / NiH+ / metal-oxide fragment",
    69: "Ga+",
    70: "FeOH+ / NiH2+ / oxide fragment",
    71: "FeOH+ / oxide fragment",
    72: "FeO2+ / CrO3+ / CaO2+",
    73: "FeO+ / CrO2+ / GaO+ / CaO2H+",
    74: "FeO2H+ / CrO3H+",
    75: "CrO3H+ / FeO3+",
    76: "CrO4+ / FeO3H+",
    77: "FeOH2+ / FeO2H+",
    78: "FeO3+ / CrO4H+",
    79: "FeO3H+ / NiO2+",
    80: "ZnO+ / SO3+ / NiO+",
    81: "ZnOH+ / FeO2OH+ / HSO3+",
    82: "ZnO+ / FeO3+",
    83: "ZnOH+ / CrO3OH+",
    84: "ZnO2+",
    85: "ZnO2H+ / FeO3H+",
    88: "Sr+ / Ca2+ candidate",
    90: "Zr+ / MoO+",
    91: "ZrH+ / MoOH+",
    92: "Mo+ / Zr+",
    93: "Nb+ / MoH+",
    94: "Mo isotope / MoO+",
    95: "Mo isotope",
    96: "Mo isotope / MoO+",
    97: "MoOH+",
    98: "Mo isotope / MoO+",
    99: "MoO+ / MoH2+",
    100: "MoO+ / Ru+",
    108: "FeNi+ / Fe2O2+ / MoO2+",
    110: "MoO2+ / ZnS+",
    111: "MoO2H+ / ZnSH+ / ZnPO2+",
    112: "Fe2+ / FeCr+ / MoS+",
    124: "Xe+ isotope / Fe2O2+ / FeCrO+",
    126: "Xe+ isotope / MoO3+ / ZnPO2+",
    128: "Xe+ isotope / Fe2O+ / CrFeO+ / MoS2+",
    129: "Xe+ isotope",
    130: "Xe+ isotope",
    131: "Xe+ isotope",
    132: "Xe+ isotope",
    134: "Xe+ isotope",
    136: "Xe+ isotope / CaSO4+ / additive salt cluster",
    144: "MoS2O+ / phosphate cluster",
    177: "ZnPO3+ / zinc phosphate cluster",
    223: "ZnP2O6+ / zinc pyrophosphate cluster"
}

possible_ion_map_negative = {
    12: "C-",
    13: "CH-",
    16: "O-",
    17: "OH-",
    19: "F-",
    24: "C2-",
    25: "C2H-",
    26: "CN- / BO-",
    27: "BOH-",
    32: "O2- / S-",
    33: "SH- / O2H-",
    35: "Cl-",
    37: "Cl- / C3H-",
    42: "BO2- / CNO-",
    43: "HBO2- / CNOH-",
    59: "BO3-",
    60: "HBO3-",
    63: "PO2-",
    72: "FeO2- / CrO3-",
    79: "PO3-",
    84: "FeO2H- / CrO2OH-",
    88: "SiO3-",
    89: "FeOH2- / oxide-hydroxide fragment",
    95: "PO4H2-",
    96: "HSO4-",
    97: "H2PO4- / H2SO4-",
    98: "H3PO4-",
    124: "Xe- candidate / heavy cluster candidate",
    126: "Xe- candidate / heavy cluster candidate",
    127: "P2O5H-",
    128: "Xe- candidate / heavy cluster candidate",
    129: "Xe- candidate",
    130: "Xe- candidate",
    131: "Xe- candidate",
    132: "Xe- candidate",
    134: "Xe- candidate",
    136: "Xe- candidate",
    143: "P2O6H-",
    146: "Cr2O3- / FeSiO3- / cluster ion",
    159: "P2O7H-"
}

# =============================
# 3. LABEL HELPERS
# =============================

def fmt_sup(sup):
    sup = sup.strip()
    if not sup:
        return ""
    sup = sup.replace(" ", "")
    if len(sup) >= 2 and sup[-1] in "+-":
        sup = sup[:-1] + r"\!" + sup[-1]
    return sup


def make_display_part(symbol, sub, sup):
    symbol = symbol.strip()
    sub = sub.strip()
    sup = fmt_sup(sup)

    if not symbol:
        return ""

    if sub and sup:
        return rf"$\mathrm{{{symbol}}}_{{{sub}}}^{{\,{sup}}}$"
    if sub:
        return rf"$\mathrm{{{symbol}}}_{{{sub}}}$"
    if sup:
        return rf"$\mathrm{{{symbol}}}^{{{sup}}}$"
    return symbol


def make_plain_part(symbol, sub, sup):
    symbol = symbol.strip()
    sub = sub.strip()
    sup = sup.strip().replace(" ", "")

    if not symbol:
        return ""

    return f"{symbol}{sub}{sup}"


def build_labels():
    parts = [
        (sym1.value, sub1.value, sup1.value),
        (sym2.value, sub2.value, sup2.value),
        (sym3.value, sub3.value, sup3.value),
        (sym4.value, sub4.value, sup4.value),
    ]

    display_label = "".join(
        make_display_part(symbol, sub, sup)
        for symbol, sub, sup in parts
        if symbol.strip()
    )

    plain_label = "".join(
        make_plain_part(symbol, sub, sup)
        for symbol, sub, sup in parts
        if symbol.strip()
    )

    return display_label, plain_label


def safe_name(text):
    if not text:
        return "ion"

    keep = []
    for ch in text:
        if ch.isalnum() or ch in ("+", "-"):
            keep.append(ch)

    return "".join(keep) or "ion"


def ion_no_3digit():
    try:
        return f"{int(ion_box.value.strip()):03d}"
    except Exception:
        return "000"

# =============================
# 4. FILE HELPERS
# =============================

def detect_polarity_from_path(path_text):
    path_lower = str(path_text).lower()
    if "positive" in path_lower:
        return "Positive"
    elif "negative" in path_lower:
        return "Negative"
    else:
        return "Unknown"


def make_display_name(file_path):
    file_path = Path(file_path)
    folder_name = file_path.parent.name
    pol = detect_polarity_from_path(file_path)
    return f"{folder_name}_{pol}"


def refresh_h5_dropdown():
    global current_working_folder

    if not current_working_folder.exists() or not current_working_folder.is_dir():
        file_dropdown.options = [("No .h5 files found", "")]
        file_dropdown.value = ""
        file_count_html.value = "<span style='color:red;'>Working folder is not valid.</span>"
        process_btn.disabled = True
        return

    h5_files = sorted(current_working_folder.rglob("*.h5"))

    if len(h5_files) == 0:
        file_dropdown.options = [("No .h5 files found", "")]
        file_dropdown.value = ""
        file_count_html.value = "<span style='color:#d97706;'>No .h5 files found in this folder.</span>"
        process_btn.disabled = True
        return

    options = []
    for p in h5_files:
        try:
            label = str(p.relative_to(current_working_folder))
        except Exception:
            label = str(p.name)
        options.append((label, str(p.resolve())))

    file_dropdown.options = options
    file_dropdown.value = options[0][1]
    file_count_html.value = f"<b>{len(options)}</b> .h5 file(s) found"
    process_btn.disabled = False


def apply_working_folder(btn=None):
    global current_working_folder

    raw_path = working_folder_box.value.strip()

    if raw_path == "":
        with status_out:
            clear_output(wait=True)
            print("Please enter a working folder path.")
        return

    path_obj = Path(raw_path).expanduser()
    current_working_folder = path_obj.resolve()
    working_folder_info_html.value = f"<b>Working folder:</b> {current_working_folder}"

    if not current_working_folder.exists():
        with status_out:
            clear_output(wait=True)
            print("Path does not exist.")
        file_dropdown.options = [("No .h5 files found", "")]
        file_dropdown.value = ""
        file_count_html.value = "<span style='color:red;'>Path does not exist.</span>"
        process_btn.disabled = True
        return

    if not current_working_folder.is_dir():
        with status_out:
            clear_output(wait=True)
            print("Path is not a folder.")
        file_dropdown.options = [("No .h5 files found", "")]
        file_dropdown.value = ""
        file_count_html.value = "<span style='color:red;'>Path is not a folder.</span>"
        process_btn.disabled = True
        return

    with status_out:
        clear_output(wait=True)

    refresh_h5_dropdown()

# =============================
# 5. DOWNLOAD HELPERS
# =============================

def dataframe_to_csv_bytes(df):
    return df.to_csv(index=False).encode("utf-8")


def figure_to_png_bytes(fig):
    for ax in fig.axes:
        ax.set_facecolor("none")
    fig.patch.set_alpha(0)

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=600, bbox_inches="tight", transparent=True)
    buf.seek(0)
    return buf.read()


def make_download_link_html(filename, data_bytes, label, color_class="blue"):
    if isinstance(data_bytes, memoryview):
        data_bytes = data_bytes.tobytes()

    if filename.lower().endswith(".png"):
        mime = "image/png"
    elif filename.lower().endswith(".csv"):
        mime = "text/csv"
    else:
        mime = "application/octet-stream"

    b64 = base64.b64encode(data_bytes).decode("utf-8")

    return f"""
    <a download="{filename}"
       href="data:{mime};base64,{b64}"
       class="download-btn {color_class}">
       {label}
    </a>
    """

# =============================
# 6. TABLE HTML RENDER
# =============================

def set_peak_table_html(df):
    if df is None or df.empty:
        peak_table_html.value = ""
        return

    styled = (
        df.style
        .hide(axis="index")
        .format({
            "m/z": "{:.3f}",
            "Intensity": "{:.0f}",
            "Cumulative Intensity": "{:.0f}",
            "Intensity %": "{:.3f}",
            "Cumulative Intensity %": "{:.3f}"
        })
        .set_table_styles([
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                    ("font-size", "14px")
                ]
            },
            {
                "selector": "thead th",
                "props": [
                    ("text-align", "center"),
                    ("padding", "6px 8px"),
                    ("border-bottom", "1px solid #999")
                ]
            },
            {
                "selector": "tbody td",
                "props": [
                    ("text-align", "center"),
                    ("padding", "6px 8px")
                ]
            },
            {
                "selector": "tbody td.col7",
                "props": [
                    ("text-align", "left"),
                    ("white-space", "normal")
                ]
            }
        ])
    )

    peak_table_html.value = styled.to_html()


def build_peak_table_figure():
    table_data = peak_table.copy()
    table_data["m/z"] = table_data["m/z"].map(lambda x: f"{x:.3f}")
    table_data["Intensity"] = table_data["Intensity"].map(lambda x: f"{x:.0f}")
    table_data["Cumulative Intensity"] = table_data["Cumulative Intensity"].map(lambda x: f"{x:.0f}")
    table_data["Intensity %"] = table_data["Intensity %"].map(lambda x: f"{x:.3f}")
    table_data["Cumulative Intensity %"] = table_data["Cumulative Intensity %"].map(lambda x: f"{x:.3f}")

    fig_h = max(1.8, 0.32 * (len(table_data) + 1))
    fig, ax = plt.subplots(figsize=(16, fig_h))
    ax.axis("off")

    fig.text(0.5, 0.81, table_title, ha="center", va="top", fontsize=14)

    table = ax.table(
        cellText=table_data.values,
        colLabels=table_data.columns,
        cellLoc="center",
        bbox=[0.00, 0.02, 1.00, 0.86]
    )

    table.auto_set_font_size(False)
    table.set_fontsize(8.5)
    table.scale(1, 1.15)

    col_widths = [0.03, 0.03, 0.05, 0.06, 0.08, 0.06, 0.08, 0.11]
    n_rows = len(table_data) + 1
    n_cols = len(table_data.columns)

    for c in range(n_cols):
        for r in range(n_rows):
            cell = table[(r, c)]
            if c < len(col_widths):
                cell.set_width(col_widths[c])

            if c == n_cols - 1:
                txt = cell.get_text()
                txt.set_wrap(True)
                txt.set_ha("left")
                txt.set_va("center")

    return fig


def refresh_table_download_controls():
    with table_downloads_output:
        clear_output(wait=True)

        if peak_table is None or peak_table.empty:
            return

        items = []

        csv_name = f"{base_name}_{polarity}_Peak_Table.csv"
        items.append(
            make_download_link_html(
                csv_name,
                dataframe_to_csv_bytes(peak_table),
                "Save Peak Table CSV",
                "orange"
            )
        )

        fig = build_peak_table_figure()
        png_bytes = figure_to_png_bytes(fig)
        plt.close(fig)

        png_name = f"{base_name}_{polarity}_Peak_Table.png"
        items.append(
            make_download_link_html(
                png_name,
                png_bytes,
                "Save Peak Table PNG",
                "gray"
            )
        )

        display(HTML(f"<div class='download-row'>{''.join(items)}</div>"))

# =============================
# 7. SPECIAL HELPERS
# =============================

def special_projection_ticks(vmax):
    if 0.018 < vmax < 0.023:
        last_tick = 0.002 * np.floor(vmax / 0.002)
        return np.arange(0.0, last_tick + 0.0001, 0.002)
    return None


def apply_front_projection_axis(ax, start_sel, finish_sel):
    ax.set_ylim(finish_sel, start_sel)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))

    if start_sel == 1:
        ticks = ax.get_yticks()
        ticks = np.asarray(ticks, dtype=float)
        if 0.0 not in ticks:
            ticks = np.append(ticks, 0.0)
        ticks = np.unique(np.round(ticks, 10))
        ax.set_yticks(np.sort(ticks))

# =============================
# 8. LABEL BUILDER
# =============================

def make_box(width_px, height_px):
    return widgets.Text(
        value="",
        placeholder="",
        description="",
        layout=widgets.Layout(
            width=f"{width_px}px",
            height=f"{height_px}px"
        )
    )


def make_group():
    canvas = widgets.GridspecLayout(
        4, 2,
        width="80px",
        height="100px",
        grid_gap="0px 0px"
    )

    sym = make_box(40, 20)
    sup = make_box(30, 15)
    sub = make_box(30, 15)

    canvas[1:3, 0] = sym
    canvas[0:1, 1] = sup
    canvas[2:3, 1] = sub

    return sym, sub, sup, canvas


sym1, sub1, sup1, g1 = make_group()
sym2, sub2, sup2, g2 = make_group()
sym3, sub3, sup3, g3 = make_group()
sym4, sub4, sup4, g4 = make_group()

label_builder = widgets.HBox(
    [g1, g2, g3, g4],
    layout=widgets.Layout(
        margin="10px 0 24px 0",
        gap="0px"
    )
)

# =============================
# 9. UI CONTROLS
# =============================

working_folder_box = widgets.Text(
    value=str(Path.cwd().resolve()),
    description="Working Folder",
    layout=widgets.Layout(width="850px"),
    style={"description_width": "110px"}
)

apply_folder_btn = widgets.Button(description="Apply Working Folder", button_style="primary")
refresh_files_btn = widgets.Button(description="Refresh file list", button_style="primary")
process_btn = widgets.Button(description="Process", button_style="success", disabled=True)

working_folder_info_html = widgets.HTML(value=f"<b>Working folder:</b> {current_working_folder}")

file_dropdown = widgets.Dropdown(
    options=[("No .h5 files found", "")],
    description="H5 file",
    layout=widgets.Layout(width="850px"),
    style={"description_width": "110px"}
)

file_count_html = widgets.HTML(value="")

file_note_html = widgets.HTML(value="")
polarity_html = widgets.HTML(value="")
peak_table_html = widgets.HTML(value="")

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Process:",
    bar_style="",
    layout=widgets.Layout(width="500px")
)

progress_label = widgets.HTML("0%", layout=widgets.Layout(width="60px"))

status_out = widgets.Output()
message_out = widgets.Output()
plot_out = widgets.Output(layout=widgets.Layout(margin="20px 0 0 0"))
table_downloads_output = widgets.Output()
plot_downloads_output = widgets.Output()

ion_box = widgets.Text(
    value="",
    placeholder="e.g. 16",
    description="Ion",
    layout=widgets.Layout(width="165px")
)

apply_ion_btn = widgets.Button(description="Apply ion")

selected_start_frame_box = widgets.Text(
    value="",
    placeholder="min. 1",
    description="Start frame",
    layout=widgets.Layout(width="200px")
)

selected_end_frame_box = widgets.Text(
    value="",
    placeholder=f"max. {n_frames_total}",
    description="End frame",
    layout=widgets.Layout(width="200px")
)

apply_selected_frames_btn = widgets.Button(description="Apply selected frames")
reset_selected_frames_btn = widgets.Button(description="Reset selected frames")

frame_slider = widgets.IntRangeSlider(
    value=[1, n_frames_total],
    min=1,
    max=n_frames_total,
    step=1,
    description="Frames",
    continuous_update=False,
    readout=False,
    layout=widgets.Layout(width="740px")
)

start_box = widgets.Text(value="1", layout=widgets.Layout(width="90px"))
finish_box = widgets.Text(value=str(n_frames_total), layout=widgets.Layout(width="90px"))

auto_depth = widgets.Checkbox(value=True, description="Auto 1")
max_depth = widgets.Text(value="0.0000", description="Max 1", layout=widgets.Layout(width="150px"))
apply_max1_btn = widgets.Button(description="Apply Max 1")

auto_tp = widgets.Checkbox(value=True, description="Auto 2")
max_tp = widgets.Text(value="0.000", description="Max 2", layout=widgets.Layout(width="150px"))
apply_max2_btn = widgets.Button(description="Apply Max 2")

auto_fp = widgets.Checkbox(value=True, description="Auto 3")
max_fp = widgets.Text(value="0.000", description="Max 3", layout=widgets.Layout(width="150px"))
apply_max3_btn = widgets.Button(description="Apply Max 3")

apply_ion_name_btn = widgets.Button(description="Apply ion name")
reset_ion_name_btn = widgets.Button(description="Reset ion name")

# =============================
# 10. CURRENT DISPLAY STATE
# =============================

current_depth_profile = None
current_tp = None
current_fp = None
current_display_label = ""
current_plain_label = ""
current_start = None
current_finish = None
current_start_orig = None
current_finish_orig = None
current_vmax_depth = None
current_vmax_tp = None
current_vmax_fp = None
current_software_ion = None

_syncing = False

# =============================
# 11. CACHE HELPERS
# =============================

def clear_analysis_caches():
    depth_profile_cache.clear()
    projection_cache.clear()


def get_cached_depth_profile(software_ion):
    key = (analysis_frame_start, analysis_frame_end, software_ion)

    if key in depth_profile_cache:
        return depth_profile_cache[key]

    ion_idx = software_ion - 1
    depth_profile = np.empty(n_frames, dtype=np.float64)

    for out_i, src_i in enumerate(range(analysis_frame_start - 1, analysis_frame_end)):
        depth_profile[out_i] = np.mean(peak[src_i, :, :, ion_idx], dtype=np.float64)

    depth_profile_cache[key] = depth_profile
    return depth_profile


def get_cached_projections(software_ion, start, finish):
    key = (analysis_frame_start, analysis_frame_end, software_ion, start, finish)

    if key in projection_cache:
        return projection_cache[key]

    ion_idx = software_ion - 1
    start_idx = (analysis_frame_start - 1) + (start - 1)
    finish_idx = (analysis_frame_start - 1) + finish

    cube = peak[start_idx:finish_idx, :, :, ion_idx]
    tp = np.mean(cube, axis=0)
    fp = np.mean(cube, axis=1)

    projection_cache[key] = (tp, fp)
    return tp, fp

# =============================
# 12. RESET / CLEAR
# =============================

def clear_runtime_state():
    global current_depth_profile
    global current_tp
    global current_fp
    global current_display_label
    global current_plain_label
    global current_start
    global current_finish
    global current_start_orig
    global current_finish_orig
    global current_vmax_depth
    global current_vmax_tp
    global current_vmax_fp
    global current_software_ion

    current_depth_profile = None
    current_tp = None
    current_fp = None
    current_display_label = ""
    current_plain_label = ""
    current_start = None
    current_finish = None
    current_start_orig = None
    current_finish_orig = None
    current_vmax_depth = None
    current_vmax_tp = None
    current_vmax_fp = None
    current_software_ion = None


def hard_reset_all():
    global f
    global peak
    global mass
    global spectrum
    global selected_file_path
    global selected_file_name
    global selected_folder_name
    global selected_display_name
    global scan_size
    global n_frames_total
    global n_elements
    global default_analysis_frame_start
    global default_analysis_frame_end
    global analysis_frame_start
    global analysis_frame_end
    global n_frames
    global frames
    global base_name
    global polarity
    global possible_ion_map
    global peak_table
    global table_title
    global _syncing

    try:
        if f is not None:
            f.close()
    except Exception:
        pass

    plt.close("all")

    f = None
    peak = None
    mass = None
    spectrum = None

    selected_file_path = None
    selected_file_name = ""
    selected_folder_name = ""
    selected_display_name = ""

    scan_size = 25.4
    n_frames_total = 1
    n_elements = 1

    default_analysis_frame_start = 1
    default_analysis_frame_end = 1
    analysis_frame_start = 1
    analysis_frame_end = 1

    n_frames = 1
    frames = np.arange(1, n_frames + 1)

    base_name = ""
    polarity = "Unknown"
    possible_ion_map = {}
    peak_table = pd.DataFrame()
    table_title = "Table of Possible Ions"

    clear_analysis_caches()
    clear_runtime_state()

    _syncing = True

    ion_box.value = ""
    selected_start_frame_box.value = ""
    selected_end_frame_box.value = ""
    selected_end_frame_box.placeholder = "max. 1"

    start_box.value = "1"
    finish_box.value = "1"

    auto_depth.value = True
    auto_tp.value = True
    auto_fp.value = True

    max_depth.value = "0.0000"
    max_tp.value = "0.000"
    max_fp.value = "0.000"

    max_depth.disabled = False
    max_tp.disabled = False
    max_fp.disabled = False

    frame_slider.min = 1
    frame_slider.max = 1
    frame_slider.value = (1, 1)

    sym1.value = ""
    sub1.value = ""
    sup1.value = ""
    sym2.value = ""
    sub2.value = ""
    sup2.value = ""
    sym3.value = ""
    sub3.value = ""
    sup3.value = ""
    sym4.value = ""
    sub4.value = ""
    sup4.value = ""

    _syncing = False

    progress_bar.value = 0
    progress_bar.bar_style = ""
    progress_label.value = "0%"

    with status_out:
        clear_output(wait=True)

    with message_out:
        clear_output(wait=True)

    with plot_out:
        clear_output(wait=False)

    with table_downloads_output:
        clear_output(wait=False)

    with plot_downloads_output:
        clear_output(wait=False)

    peak_table_html.value = ""
    polarity_html.value = ""
    file_note_html.value = ""

    controls_box.layout.display = "none"

# =============================
# 13. HELPERS FOR TEXT BOXES
# =============================

def parse_int_text(box, name):
    try:
        return int(box.value.strip())
    except Exception:
        raise ValueError(f"{name} must be an integer.")


def parse_float_text(box, name):
    try:
        return float(box.value.strip())
    except Exception:
        raise ValueError(f"{name} must be a number.")


def parse_selected_frame_range():
    try:
        start_sel = int(selected_start_frame_box.value.strip())
    except Exception:
        raise ValueError("Start frame must be an integer.")

    try:
        end_sel = int(selected_end_frame_box.value.strip())
    except Exception:
        raise ValueError("End frame must be an integer.")

    if start_sel < 1:
        raise ValueError("Start frame must be at least 1.")

    if end_sel > n_frames_total:
        raise ValueError(f"End frame must be at most {n_frames_total}.")

    if end_sel < start_sel:
        raise ValueError("End frame must be greater than or equal to Start frame.")

    return start_sel, end_sel

# =============================
# 14. FILE PREP
# =============================

def build_peak_table_from_current_file():
    global peak_table
    global table_title

    peak_idx, _ = find_peaks(spectrum, height=100)

    peak_mz = mass[peak_idx]
    peak_intensity = spectrum[peak_idx]

    nominal_ions = np.rint(peak_mz).astype(int)
    valid_mask = (nominal_ions >= 1) & (nominal_ions <= n_elements)

    peak_mz = peak_mz[valid_mask]
    peak_intensity = peak_intensity[valid_mask]
    nominal_ions = nominal_ions[valid_mask]

    best_peaks = {}
    for mz_val, inten_val, ion_val in zip(peak_mz, peak_intensity, nominal_ions):
        if ion_val not in best_peaks or inten_val > best_peaks[ion_val]["Intensity"]:
            best_peaks[ion_val] = {
                "Ion": ion_val,
                "m/z": mz_val,
                "Intensity": inten_val
            }

    peak_table = pd.DataFrame(best_peaks.values())
    peak_table = peak_table.sort_values("Intensity", ascending=False).reset_index(drop=True)
    peak_table.insert(0, "Rank", np.arange(1, len(peak_table) + 1))

    total_intensity = peak_table["Intensity"].sum()
    peak_table["Cumulative Intensity"] = peak_table["Intensity"].cumsum().round(0)
    peak_table["Intensity %"] = (100 * peak_table["Intensity"] / total_intensity).round(3)
    peak_table["Cumulative Intensity %"] = peak_table["Intensity %"].cumsum().round(3)
    peak_table["Possible ion"] = peak_table["Ion"].map(
        lambda x: possible_ion_map.get(x, "Unknown / needs check")
    )

    peak_table = peak_table[
        [
            "Rank",
            "Ion",
            "m/z",
            "Intensity",
            "Cumulative Intensity",
            "Intensity %",
            "Cumulative Intensity %",
            "Possible ion"
        ]
    ]

    if polarity == "Positive":
        table_title = "Table of Possible Positive Ions"
    elif polarity == "Negative":
        table_title = "Table of Possible Negative Ions"
    else:
        table_title = "Table of Possible Ions"

# =============================
# 15. PROCESS FILE
# =============================

def process_selected_file(btn=None):
    global f
    global peak
    global mass
    global spectrum
    global scan_size
    global n_frames_total
    global n_elements
    global default_analysis_frame_start
    global default_analysis_frame_end
    global analysis_frame_start
    global analysis_frame_end
    global n_frames
    global frames
    global base_name
    global polarity
    global possible_ion_map
    global _syncing
    global selected_file_path
    global selected_file_name
    global selected_folder_name
    global selected_display_name

    file_path_value = file_dropdown.value

    if not file_path_value:
        with status_out:
            clear_output(wait=True)
            print("No .h5 file selected.")
        return

    hard_reset_all()

    try:
        progress_bar.value = 10
        progress_bar.bar_style = "info"
        progress_label.value = "10%"

        selected_file_path = Path(file_path_value).resolve()
        selected_file_name = selected_file_path.name
        selected_folder_name = selected_file_path.parent.name
        selected_display_name = make_display_name(selected_file_path)

        file_note_html.value = f"<b>{selected_file_path}</b>"

        progress_bar.value = 30
        progress_label.value = "30%"

        f = h5py.File(selected_file_path, "r")

        progress_bar.value = 50
        progress_label.value = "50%"

        peak = f["PeakData"]["PeakData"]
        mass = f["FullSpectra"]["MassAxis"][:]
        spectrum = f["FullSpectra"]["SumSpectrum"][:]

        scan_size = 25.4
        n_frames_total = peak.shape[0]
        n_elements = peak.shape[3]

        default_analysis_frame_start = 1
        default_analysis_frame_end = n_frames_total

        analysis_frame_start = default_analysis_frame_start
        analysis_frame_end = default_analysis_frame_end

        n_frames = analysis_frame_end - analysis_frame_start + 1
        frames = np.arange(1, n_frames + 1)

        base_name = selected_file_path.stem
        polarity = detect_polarity_from_path(selected_file_path)

        if polarity == "Positive":
            possible_ion_map = possible_ion_map_positive
        elif polarity == "Negative":
            possible_ion_map = possible_ion_map_negative
        else:
            possible_ion_map = {}

        progress_bar.value = 70
        progress_label.value = "70%"

        build_peak_table_from_current_file()
        set_peak_table_html(peak_table)
        refresh_table_download_controls()
        polarity_html.value = f"<b>Detected Polarity:</b> {polarity}"

        selected_end_frame_box.placeholder = f"max. {n_frames_total}"

        _syncing = True
        start_box.value = "1"
        finish_box.value = str(n_frames_total)
        frame_slider.min = 1
        frame_slider.max = n_frames_total
        frame_slider.value = (1, n_frames_total)
        _syncing = False

        clear_runtime_state()
        controls_box.layout.display = ""

        progress_bar.value = 100
        progress_bar.bar_style = "success"
        progress_label.value = "Done"

        with status_out:
            clear_output(wait=True)

        with message_out:
            clear_output(wait=True)
            print(f"{selected_display_name} h5 file processed")

    except Exception as e:
        progress_bar.bar_style = "danger"
        progress_label.value = "error"
        with status_out:
            clear_output(wait=True)
            print(f"Error: {e}")

# =============================
# 16. FIGURES FOR DOWNLOAD LINKS
# =============================

def build_depth_profile_save_figure():
    fig, ax = plt.subplots(figsize=(save_width_depth, save_height))
    ax.plot(frames, current_depth_profile)

    show_span = not (current_start == 1 and current_finish == n_frames)
    if show_span:
        ax.axvspan(current_start, current_finish, alpha=0.2)

    ax.set_xlim(0, n_frames)
    ticks = ax.get_xticks()
    ticks = np.unique(np.append(ticks, 0))
    ax.set_xticks(np.sort(ticks))
    ax.set_ylim(0, current_vmax_depth)
    ax.set_xlabel("Frame number")
    ax.set_ylabel("Intensity")
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))
    ax.tick_params(direction="out")
    ax.set_title(f"Depth Profile - {current_display_label}")
    return fig


def build_tp_save_figure():
    fig, ax = plt.subplots(figsize=(save_width_tp, save_height))
    im = ax.imshow(
        current_tp,
        cmap="jet",
        interpolation="bilinear",
        extent=[0, scan_size, scan_size, 0],
        vmin=0,
        vmax=current_vmax_tp
    )

    ax.set_box_aspect(1)
    ax.set_xlabel("X [µm]")
    ax.set_ylabel("Y [µm]")
    ax.set_xticks([0, 5, 10, 15, 20, 25])
    ax.set_yticks([0, 5, 10, 15, 20, 25])
    ax.tick_params(direction="out")
    ax.set_title(f"Top Projection - {current_display_label}")

    cbar = fig.colorbar(im, ax=ax, format="%.3f")
    ticks = special_projection_ticks(current_vmax_tp)
    if ticks is not None:
        cbar.locator = FixedLocator(ticks)
        cbar.formatter = FormatStrFormatter("%.3f")
    cbar.set_label("Intensity")
    cbar.update_ticks()
    return fig


def build_fp_save_figure():
    fig, ax = plt.subplots(figsize=(save_width_fp, save_height))
    im = ax.imshow(
        current_fp,
        cmap="jet",
        interpolation="bilinear",
        extent=[0, scan_size, current_finish_orig, current_start_orig],
        aspect="auto",
        vmin=0,
        vmax=current_vmax_fp
    )

    ax.set_box_aspect(1)
    ax.set_xlabel("X [µm]")
    ax.set_ylabel("Frame number")
    ax.set_xticks([0, 5, 10, 15, 20, 25])
    ax.tick_params(direction="out")
    ax.set_title(f"Front Projection - {current_display_label}")
    apply_front_projection_axis(ax, current_start, current_finish)

    cbar = fig.colorbar(im, ax=ax, format="%.3f")
    ticks = special_projection_ticks(current_vmax_fp)
    if ticks is not None:
        cbar.locator = FixedLocator(ticks)
        cbar.formatter = FormatStrFormatter("%.3f")
    cbar.set_label("Intensity")
    cbar.update_ticks()
    return fig


def refresh_plot_download_controls():
    with plot_downloads_output:
        clear_output(wait=True)

        if current_depth_profile is None or current_tp is None or current_fp is None:
            return

        no_ions = ion_no_3digit()
        items = []

        depth_csv_name = f"{base_name}_{no_ions}_{current_plain_label}_Depth_Profile_{current_start}_{current_finish}.csv"
        depth_df = pd.DataFrame({
            "Frame": frames,
            "Intensity": current_depth_profile
        })
        items.append(
            make_download_link_html(
                depth_csv_name,
                dataframe_to_csv_bytes(depth_df),
                "Save Depth Profile CSV",
                "orange"
            )
        )

        fig_depth = build_depth_profile_save_figure()
        depth_png_name = f"{base_name}_{no_ions}_{current_plain_label}_Depth_Profile_{current_start}_{current_finish}.png"
        items.append(
            make_download_link_html(
                depth_png_name,
                figure_to_png_bytes(fig_depth),
                "Save Depth Profile PNG",
                "gray"
            )
        )
        plt.close(fig_depth)

        fig_tp = build_tp_save_figure()
        tp_png_name = f"{base_name}_{no_ions}_{current_plain_label}_Top_Projection_{current_start}_{current_finish}.png"
        items.append(
            make_download_link_html(
                tp_png_name,
                figure_to_png_bytes(fig_tp),
                "Save TP Image",
                "gray"
            )
        )
        plt.close(fig_tp)

        fig_fp = build_fp_save_figure()
        fp_png_name = f"{base_name}_{no_ions}_{current_plain_label}_Front_Projection_{current_start}_{current_finish}.png"
        items.append(
            make_download_link_html(
                fp_png_name,
                figure_to_png_bytes(fig_fp),
                "Save FP Image",
                "gray"
            )
        )
        plt.close(fig_fp)

        display(HTML(f"<div class='download-row'>{''.join(items)}</div>"))

# =============================
# 17. CORE UPDATE HELPERS
# =============================

def apply_visual_settings_to_current_data(software_ion, depth_profile, tp, fp, start, finish):
    global current_depth_profile
    global current_tp
    global current_fp
    global current_display_label
    global current_plain_label
    global current_start
    global current_finish
    global current_start_orig
    global current_finish_orig
    global current_vmax_depth
    global current_vmax_tp
    global current_vmax_fp
    global current_software_ion

    display_label, plain_label = build_labels()

    title_label = display_label if display_label else f"Ion {software_ion}"
    file_label = safe_name(plain_label) if plain_label else f"Ion{software_ion}"

    vmax1_data = float(np.max(depth_profile)) if np.max(depth_profile) > 0 else 1.0
    vmax2_data = float(np.max(tp)) if np.max(tp) > 0 else 1.0
    vmax3_data = float(np.max(fp)) if np.max(fp) > 0 else 1.0

    if auto_depth.value:
        vmax1 = vmax1_data
        max_depth.value = f"{vmax1:.3f}"
        max_depth.disabled = True
    else:
        max_depth.disabled = False
        vmax1 = parse_float_text(max_depth, "Max 1")
        if vmax1 <= 0:
            vmax1 = vmax1_data

    if auto_tp.value:
        vmax2 = vmax2_data
        max_tp.value = f"{vmax2:.3f}"
        max_tp.disabled = True
    else:
        max_tp.disabled = False
        vmax2 = parse_float_text(max_tp, "Max 2")
        if vmax2 <= 0:
            vmax2 = vmax2_data

    if auto_fp.value:
        vmax3 = vmax3_data
        max_fp.value = f"{vmax3:.3f}"
        max_fp.disabled = True
    else:
        max_fp.disabled = False
        vmax3 = parse_float_text(max_fp, "Max 3")
        if vmax3 <= 0:
            vmax3 = vmax3_data

    current_depth_profile = depth_profile
    current_tp = tp
    current_fp = fp
    current_display_label = title_label
    current_plain_label = file_label
    current_start = start
    current_finish = finish
    current_start_orig = start
    current_finish_orig = finish
    current_vmax_depth = vmax1
    current_vmax_tp = vmax2
    current_vmax_fp = vmax3
    current_software_ion = software_ion


def redraw_current_plots():
    if current_depth_profile is None or current_tp is None or current_fp is None:
        with plot_out:
            clear_output(wait=False)
        with plot_downloads_output:
            clear_output(wait=False)
        return

    with plot_out:
        clear_output(wait=False)

        fig, axes = plt.subplots(
            3, 1,
            figsize=(7, 14),
            gridspec_kw={"height_ratios": [1, 1.2, 1.2]}
        )

        ax1, ax2, ax3 = axes

        ax1.plot(frames, current_depth_profile)

        show_span = not (current_start == 1 and current_finish == n_frames)
        if show_span:
            ax1.axvspan(current_start, current_finish, alpha=0.2)

        ax1.set_xlim(0, n_frames)
        ticks = ax1.get_xticks()
        ticks = np.unique(np.append(ticks, 0))
        ax1.set_xticks(np.sort(ticks))
        ax1.set_ylim(0, current_vmax_depth)
        ax1.set_xlabel("Frame number")
        ax1.set_ylabel("Intensity")
        ax1.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))
        ax1.tick_params(direction="out")
        ax1.set_title(f"Depth Profile - {current_display_label}")

        im2 = ax2.imshow(
            current_tp,
            cmap="jet",
            interpolation="bilinear",
            extent=[0, scan_size, scan_size, 0],
            vmin=0,
            vmax=current_vmax_tp
        )
        ax2.set_box_aspect(1)
        ax2.set_xlabel("X [µm]")
        ax2.set_ylabel("Y [µm]")
        ax2.set_xticks([0, 5, 10, 15, 20, 25])
        ax2.set_yticks([0, 5, 10, 15, 20, 25])
        ax2.tick_params(direction="out")
        ax2.set_title(f"Top Projection - {current_display_label}")

        cbar2 = fig.colorbar(im2, ax=ax2, format="%.3f")
        ticks2 = special_projection_ticks(current_vmax_tp)
        if ticks2 is not None:
            cbar2.locator = FixedLocator(ticks2)
            cbar2.formatter = FormatStrFormatter("%.3f")
        cbar2.set_label("Intensity")
        cbar2.update_ticks()

        im3 = ax3.imshow(
            current_fp,
            cmap="jet",
            interpolation="bilinear",
            extent=[0, scan_size, current_finish_orig, current_start_orig],
            aspect="auto",
            vmin=0,
            vmax=current_vmax_fp
        )
        ax3.set_box_aspect(1)
        ax3.set_xlabel("X [µm]")
        ax3.set_ylabel("Frame number")
        ax3.set_xticks([0, 5, 10, 15, 20, 25])
        ax3.tick_params(direction="out")
        ax3.set_title(f"Front Projection - {current_display_label}")
        apply_front_projection_axis(ax3, current_start, current_finish)

        cbar3 = fig.colorbar(im3, ax=ax3, format="%.3f")
        ticks3 = special_projection_ticks(current_vmax_fp)
        if ticks3 is not None:
            cbar3.locator = FixedLocator(ticks3)
            cbar3.formatter = FormatStrFormatter("%.3f")
        cbar3.set_label("Intensity")
        cbar3.update_ticks()

        plt.tight_layout()
        plt.show()

    refresh_plot_download_controls()

# =============================
# 18. FULL UPDATE
# =============================

def update_full(change=None):
    if peak is None:
        with plot_out:
            clear_output(wait=False)
        with plot_downloads_output:
            clear_output(wait=False)
        return

    with message_out:
        clear_output(wait=True)

    text = ion_box.value.strip()

    if text == "":
        with plot_out:
            clear_output(wait=False)
        with plot_downloads_output:
            clear_output(wait=False)
        return

    try:
        software_ion = int(text)
    except ValueError:
        with message_out:
            print("Ion must be an integer.")
        return

    if software_ion < 1 or software_ion > n_elements:
        with message_out:
            print(f"Ion must be between 1 and {n_elements}.")
        return

    try:
        start = parse_int_text(start_box, "Start")
        finish = parse_int_text(finish_box, "Finish")
    except ValueError as e:
        with message_out:
            print(str(e))
        return

    if start < 1 or finish > n_frames or finish <= start:
        with message_out:
            print(f"Use 1 <= Start < Finish <= {n_frames}")
        return

    try:
        depth_profile = get_cached_depth_profile(software_ion)
        tp, fp = get_cached_projections(software_ion, start, finish)
        apply_visual_settings_to_current_data(software_ion, depth_profile, tp, fp, start, finish)
    except ValueError as e:
        with message_out:
            print(str(e))
        return
    except Exception as e:
        with message_out:
            print(f"Error: {e}")
        return

    redraw_current_plots()

# =============================
# 19. VISUAL-ONLY UPDATE
# =============================

def redraw_only(change=None):
    if current_depth_profile is None or current_tp is None or current_fp is None or current_software_ion is None:
        update_full()
        return

    with message_out:
        clear_output(wait=True)

    try:
        apply_visual_settings_to_current_data(
            current_software_ion,
            current_depth_profile,
            current_tp,
            current_fp,
            current_start,
            current_finish
        )
    except ValueError as e:
        with message_out:
            print(str(e))
        return
    except Exception as e:
        with message_out:
            print(f"Error: {e}")
        return

    redraw_current_plots()

# =============================
# 20. SYNC FUNCTIONS
# =============================

def sync_from_slider(change=None):
    global _syncing

    if _syncing:
        return

    _syncing = True
    start_box.value = str(frame_slider.value[0])
    finish_box.value = str(frame_slider.value[1])
    _syncing = False

    update_full()


def sync_from_boxes(change=None):
    global _syncing

    if _syncing:
        return

    try:
        start = parse_int_text(start_box, "Start")
        finish = parse_int_text(finish_box, "Finish")
    except ValueError as e:
        with message_out:
            clear_output(wait=True)
            print(str(e))
        return

    if start < 1:
        start = 1

    if finish > n_frames:
        finish = n_frames

    if finish <= start:
        finish = min(n_frames, start + 1)

    _syncing = True
    start_box.value = str(start)
    finish_box.value = str(finish)
    frame_slider.value = (start, finish)
    _syncing = False

    update_full()


def apply_selected_frames(change=None):
    global analysis_frame_start
    global analysis_frame_end
    global n_frames
    global frames
    global _syncing

    with message_out:
        clear_output(wait=True)

    try:
        start_sel, end_sel = parse_selected_frame_range()
    except ValueError as e:
        with message_out:
            print(str(e))
        return

    analysis_frame_start = start_sel
    analysis_frame_end = end_sel

    n_frames = analysis_frame_end - analysis_frame_start + 1
    frames = np.arange(1, n_frames + 1)

    _syncing = True
    start_box.value = "1"
    finish_box.value = str(n_frames)
    frame_slider.min = 1
    frame_slider.max = n_frames
    frame_slider.value = (1, n_frames)
    _syncing = False

    clear_runtime_state()

    with plot_out:
        clear_output(wait=False)

    with plot_downloads_output:
        clear_output(wait=False)

    with message_out:
        print(f"Using original frames {start_sel}-{end_sel} for analysis.")

    update_full()


def reset_selected_frames(change=None):
    global analysis_frame_start
    global analysis_frame_end
    global n_frames
    global frames
    global _syncing

    analysis_frame_start = default_analysis_frame_start
    analysis_frame_end = default_analysis_frame_end

    selected_start_frame_box.value = ""
    selected_end_frame_box.value = ""

    n_frames = analysis_frame_end - analysis_frame_start + 1
    frames = np.arange(1, n_frames + 1)

    _syncing = True
    start_box.value = "1"
    finish_box.value = str(n_frames_total)
    frame_slider.min = 1
    frame_slider.max = n_frames_total
    frame_slider.value = (1, n_frames_total)
    _syncing = False

    clear_runtime_state()

    with message_out:
        clear_output(wait=True)
        print(f"Reset to original frames {default_analysis_frame_start}-{default_analysis_frame_end}.")

    with plot_out:
        clear_output(wait=False)

    with plot_downloads_output:
        clear_output(wait=False)

    update_full()


def apply_ion(change=None):
    update_full()


def apply_ion_name(change=None):
    redraw_only()


def reset_ion_name(change=None):
    sym1.value = ""
    sub1.value = ""
    sup1.value = ""
    sym2.value = ""
    sub2.value = ""
    sup2.value = ""
    sym3.value = ""
    sub3.value = ""
    sup3.value = ""
    sym4.value = ""
    sub4.value = ""
    sup4.value = ""
    redraw_only()


def apply_max1(change=None):
    redraw_only()


def apply_max2(change=None):
    redraw_only()


def apply_max3(change=None):
    redraw_only()


def reset_frames(change=None):
    global _syncing

    _syncing = True
    start_box.value = "1"
    finish_box.value = str(n_frames)
    frame_slider.value = (1, n_frames)
    _syncing = False

    update_full()

# =============================
# 21. CONNECT
# =============================

apply_folder_btn.on_click(apply_working_folder)
refresh_files_btn.on_click(lambda btn: refresh_h5_dropdown())
process_btn.on_click(process_selected_file)

frame_slider.observe(sync_from_slider, names="value")

apply_frames_btn = widgets.Button(description="Apply frames")
reset_frames_btn = widgets.Button(description="Reset frames")

apply_selected_frames_btn.on_click(apply_selected_frames)
reset_selected_frames_btn.on_click(reset_selected_frames)
apply_frames_btn.on_click(sync_from_boxes)
reset_frames_btn.on_click(reset_frames)

apply_ion_btn.on_click(apply_ion)
apply_ion_name_btn.on_click(apply_ion_name)
reset_ion_name_btn.on_click(reset_ion_name)
apply_max1_btn.on_click(apply_max1)
apply_max2_btn.on_click(apply_max2)
apply_max3_btn.on_click(apply_max3)

for w in [auto_depth, auto_tp, auto_fp]:
    w.observe(redraw_only, names="value")

# =============================
# 22. DISPLAY
# =============================

controls_box = widgets.VBox([
    widgets.HBox([
        selected_start_frame_box,
        selected_end_frame_box,
        apply_selected_frames_btn,
        reset_selected_frames_btn
    ]),
    widgets.HBox([ion_box, apply_ion_btn]),
    widgets.HBox([frame_slider, start_box, finish_box, apply_frames_btn, reset_frames_btn]),
    widgets.HBox([
        widgets.HTML("<b>Ion Name</b>"),
        apply_ion_name_btn,
        reset_ion_name_btn
    ]),
    label_builder,
    widgets.HBox([auto_depth, max_depth, apply_max1_btn]),
    widgets.HBox([auto_tp, max_tp, apply_max2_btn]),
    widgets.HBox([auto_fp, max_fp, apply_max3_btn]),
    plot_out,
    plot_downloads_output,
    message_out
])

controls_box.layout.display = "none"

display(
    widgets.VBox([
        working_folder_box,
        widgets.HBox([apply_folder_btn, refresh_files_btn, process_btn]),
        working_folder_info_html,
        file_dropdown,
        file_count_html,
        widgets.HBox([progress_bar, progress_label]),
        file_note_html,
        peak_table_html,
        table_downloads_output,
        polarity_html,
        status_out,
        widgets.HTML("<div style='height:20px;'></div>"),
        controls_box
    ])
)

# fill first
apply_working_folder()